# SoukAI – Exploratory Data Analysis

This notebook performs an initial exploratory analysis of the SoukAI sample dataset for Moroccan COD e-commerce products and their associated customer comments.

**Dataset:**
- `data/products.csv` – 15 products with trend, competition and margin scores
- `data/comments.csv` – 85 customer comments in Darija, French and Arabic

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

products = pd.read_csv('../data/products.csv')
comments = pd.read_csv('../data/comments.csv')

print('Products shape:', products.shape)
print('Comments shape:', comments.shape)

## 1. Products Overview

In [ ]:
products.head()

In [ ]:
products.describe()

## 2. Category Distribution

In [ ]:
category_counts = products['category'].value_counts()

fig, ax = plt.subplots()
category_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Number of Products per Category')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. Trend Growth vs Competition Level

In [ ]:
fig, ax = plt.subplots()

scatter = ax.scatter(
    products['competition_level'],
    products['trend_growth'],
    c=products['estimated_profit_margin'],
    s=products['price'] / 2,
    cmap='RdYlGn',
    alpha=0.8,
    edgecolors='grey',
    linewidths=0.5
)

plt.colorbar(scatter, ax=ax, label='Profit Margin')

for _, row in products.iterrows():
    ax.annotate(
        row['name'][:15],
        (row['competition_level'], row['trend_growth']),
        fontsize=7, alpha=0.7
    )

ax.set_xlabel('Competition Level (lower is better)')
ax.set_ylabel('Trend Growth (higher is better)')
ax.set_title('Product Opportunity Map\n(bubble size = price, colour = margin)')
plt.tight_layout()
plt.show()

## 4. Top Products by Trend Growth

In [ ]:
top = products.nlargest(5, 'trend_growth')[['name', 'trend_growth', 'competition_level', 'estimated_profit_margin']]
top.set_index('name').plot(kind='bar', ax=plt.subplots()[1])
plt.title('Top 5 Products by Trend Growth')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
top

## 5. Comments Analysis

In [ ]:
comments.head(10)

In [ ]:
lang_counts = comments['language'].value_counts()

fig, ax = plt.subplots()
lang_counts.plot(kind='pie', ax=ax, autopct='%1.0f%%', startangle=140,
                 colors=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax.set_ylabel('')
ax.set_title('Comment Language Distribution')
plt.tight_layout()
plt.show()

In [ ]:
comments_per_product = comments.groupby('product_id').size().reset_index(name='comment_count')
comments_per_product = comments_per_product.merge(products[['id', 'name']], left_on='product_id', right_on='id')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(comments_per_product['name'], comments_per_product['comment_count'], color='coral', edgecolor='white')
ax.set_title('Number of Comments per Product')
ax.set_ylabel('Comments')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Composite Score (SoukAI Heuristic)

A simple heuristic score: `score = 0.4 * trend_growth + 0.3 * (1 - competition_level) + 0.3 * estimated_profit_margin`

In [ ]:
products['score'] = (
    0.4 * products['trend_growth']
    + 0.3 * (1 - products['competition_level'])
    + 0.3 * products['estimated_profit_margin']
)

ranked = products[['name', 'category', 'score']].sort_values('score', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(ranked['name'], ranked['score'], color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Composite Score')
ax.set_title('SoukAI Product Ranking (Composite Score)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

ranked